# Thread safety examples

These examples mirror the thread safety examples from the slides. Each example
computes the sum of the integers from `1` to `N` using several Python threads.

The first example is intentionally unsafe because multiple threads update the
same shared variable. The second example fixes the race with a lock. The third
example avoids the shared update by giving each thread its own subtotal.

## Example 1: shared state without a lock

This version is not thread safe. The update to `total` reads, modifies, and
writes shared state, so one thread can overwrite work from another thread.
This might still give the correct answer, but it is not guaranteed.

In [ ]:
import threading
import time

N = 200_000
chunk_size = 50_000
total = 0

ranges = [(start, min(start + chunk_size, N + 1))
          for start in range(1, N + 1, chunk_size)]

def sum_chunk(start, stop):
    global total
    for i in range(start, stop):
        current_total = total
        time.sleep(0)
        total = current_total + i


threads = [threading.Thread(target=sum_chunk, args=r)
           for r in ranges]

for t in threads: t.start()
for t in threads: t.join()

print(total)

## Example 2: shared state with a lock

This version is thread safe because only one thread can update `total` while the
lock is held. The tradeoff is that the update is serialized.

In [ ]:
import threading

N = 200_000
chunk_size = 50_000
total = 0
lock = threading.Lock()

ranges = [(start, min(start + chunk_size, N + 1))
          for start in range(1, N + 1, chunk_size)]

def sum_chunk(start, stop):
    global total
    for i in range(start, stop):
        with lock:
            total += i


threads = [threading.Thread(target=sum_chunk, args=r)
           for r in ranges]

for t in threads: t.start()
for t in threads: t.join()

print(total)

## Example 3: one subtotal per thread

This version is thread safe because each thread writes to a different entry in
`subtotals`. The final summation happens after all threads have joined.

In [ ]:
import threading

N = 200_000
chunk_size = 50_000

ranges = [(start, min(start + chunk_size, N + 1))
          for start in range(1, N + 1, chunk_size)]

subtotals = [0] * len(ranges)


def sum_chunk(k, start, stop):
    for i in range(start, stop):
        subtotals[k] += i


threads = [threading.Thread(target=sum_chunk, args=(k, *r))
           for k, r in enumerate(ranges)]

for t in threads: t.start()
for t in threads: t.join()

total = sum(subtotals)

print(total)